# Dimensionality Reduction — Method Comparison
## Machine learning for energy applications — Module 4, Exercise 4

In this exercise we bring together three dimensionality-reduction methods — **PCA**,
**t-SNE**, and **UMAP** — and compare them on the petrophysical dataset under one protocol:
the same rows, the same scaling, fixed seeds and recorded settings for every method.

We evaluate:
1. Visual quality of 2D embeddings
2. Computational time (on this machine, for these 1,000 rows)
3. Neighbourhood preservation (trustworthiness at k = 10) and stability across seeds and settings
4. How separated the known facies look in each map (silhouette score)
5. Trade-offs and practical guidance

> **Exploratory analysis.** The facies labels are kept out of the inputs; they are used only to
> colour and score the maps after fitting. All fits use all 1,000 rows, so the results describe
> this table, not a model for new wells.

---
## 0 — Imports and Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Try importing UMAP (optional package: pip install umap-learn)
# (Without ipywidgets, importing umap may print a harmless TqdmWarning about IProgress.)
try:
    import umap
    UMAP_AVAILABLE = True
    print("umap-learn is available.")
except ImportError:
    UMAP_AVAILABLE = False
    print("WARNING: umap-learn is NOT installed.")
    print("Install with: pip install umap-learn")
    print("UMAP cells will be skipped.")

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Prepare the Data

We use the petrophysical dataset with 10 log channels and a facies label (integer codes 0, 1
and 2; the file gives no facies names). The copy of `petro.csv` in this folder is identical to
the one in `03_umap`.

In [ ]:
# YOUR CODE HERE
# TODO: Load "petro.csv" (the copy in this folder) into a DataFrame
# TODO: Print the shape and display the first few rows

In [ ]:
# YOUR CODE HERE
# TODO: Separate feature columns (all except "facies") and the facies label
#       (the label must not be an input of any method)
# TODO: Standardize features with StandardScaler
# TODO: Print number of features, samples, and classes

---
## 2 — Apply All Three Methods

We time each method and store the 2D embeddings for comparison. Every method gets the same
scaled matrix, seeds are fixed, and the settings are written in the code. Times are for this
machine and these 1,000 rows only; the first UMAP call also includes one-off compilation, so it
overstates the cost of later runs. Each map is then scored with trustworthiness at k = 10.

In [ ]:
# YOUR CODE HERE
# TODO: Create a dictionary `results` to store embeddings and times
#
# --- PCA ---
# TODO: Time PCA(n_components=2, random_state=RANDOM_STATE) using time.time()
# TODO: Store embedding and time in results["PCA"]
# TODO: Print time and explained variance
#
# --- t-SNE ---
# TODO: Time TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, max_iter=1000)
# TODO: Store embedding and time in results["t-SNE"]
# TODO: Print time and KL divergence
#
# --- UMAP ---
# TODO: If UMAP_AVAILABLE, time umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
#                                         random_state=RANDOM_STATE, n_jobs=1)
# TODO: Store embedding and time in results["UMAP"]
# TODO: Print time
#
# Hint: use time.time() before and after each fit_transform
#
# TODO: After the timed blocks, add results[method]["trust"] =
#       trustworthiness(X_scaled, embedding, n_neighbors=10) for every method and print it

---
## 3 — Visualize All Methods Side by Side

In [ ]:
# YOUR CODE HERE
# TODO: Create a 1 x n_methods subplot figure
# TODO: For each method, scatter the embedding coloured by facies (y)
# TODO: Include method name, time and trustworthiness in each subplot title
# TODO: Add a shared colorbar
# Hint: call plt.tight_layout(rect=[0, 0, 0.93, 1]) first, then
#       fig.add_axes for a manual colorbar axis

### 3.1 Stability: Two Seeds and Two Neighbourhood Settings

The protocol asks for at least two seeds and two neighbourhood settings for each non-linear
method. We use `init="random"` for t-SNE, so that the seed actually changes its starting layout.
Compare what changes between the runs (arrangement of the groups) with what does not
(trustworthiness).

In [ ]:
# YOUR CODE HERE
# TODO: For seeds 0 and 1:
#   - fit TSNE(n_components=2, perplexity=p, init="random", random_state=seed, max_iter=1000)
#     for p in (15, 50)
#   - if UMAP_AVAILABLE, fit umap.UMAP(n_components=2, n_neighbors=nn, min_dist=0.1,
#     random_state=seed, n_jobs=1) for nn in (15, 50)
# TODO: Compute trustworthiness(X_scaled, embedding, n_neighbors=10) for every map
# TODO: Plot all maps in a grid (coloured by facies) with the settings and trustworthiness in
#       each title. What changed between the seeds, and what did not?

---
## 4 — Computational Time Comparison

In [ ]:
# YOUR CODE HERE
# TODO: Create a bar chart comparing computation times
# TODO: Annotate each bar with the time value


---
## 5 — Facies Separation in Each Map (Silhouette Score)

The **silhouette score** of the known facies labels in each 2D map measures how separated
the facies *look* in that map. Maps can exaggerate separation (t-SNE and UMAP in particular),
so a higher map score is not evidence that the facies are separable in the logs: compare it
with the silhouette in the original 10D space (Section 6), and confirm any separation in the
logs themselves.

Note: the facies labels are used only for this check after fitting; they were never an input
of any method.

In [ ]:
# YOUR CODE HERE
# TODO: For each method, compute silhouette_score(embedding, y)
# TODO: Store in a dictionary sil_scores
# TODO: Print each score


In [ ]:
# YOUR CODE HERE
# TODO: Create a bar chart of silhouette scores
# TODO: Annotate each bar with the score value


---
## 6 — Silhouette Score in Original High-Dimensional Space

A map score far above the 10D score means the map makes the facies look more separated than
they are in the logs.

In [ ]:
# YOUR CODE HERE
# TODO: Compute silhouette_score(X_scaled, y) for the original space
# TODO: Print it and compare with each method's 2D score (show delta)


---
## 7 — Summary Comparison Table

In [ ]:
# YOUR CODE HERE
# TODO: Build a DataFrame with columns:
#   Method, Time (s), Trustworthiness T(10), Silhouette (2D), Type (Linear/Non-linear),
#   Transform new rows (Yes/No), Seed-dependent (Yes/No), Scalability
# TODO: Display the summary table
# Hint: PCA is linear, not seed-dependent, can transform new rows, excellent scalability
#       t-SNE is non-linear, seed-dependent, cannot transform new rows, moderate scalability
#       UMAP is non-linear, seed-dependent, can transform new rows, good scalability

---
## 8 — PCA Explained Variance Analysis

One advantage unique to PCA is the explained variance — we can quantify exactly
how much of the total variance is retained (variance, which is not necessarily the
information a task needs).

In [ ]:
# YOUR CODE HERE
# TODO: Fit PCA with all components
# TODO: Plot cumulative explained variance with 90% and 95% threshold lines
# TODO: Print the cumulative variance for each number of components


---
## 9 — Discussion of Trade-offs

### PCA
- **Pros**: Fast, deterministic, interpretable (loadings), has inverse transform,
  quantifiable retained variance (explained variance), scales to very large datasets.
- **Cons**: Linear — cannot capture non-linear manifold structure. May not separate
  classes well when the decision boundary is non-linear.
- **Use when**: You need a fast, interpretable first pass; preprocessing for other algorithms
  (fitted on training data only, inside a pipeline, when a model is evaluated);
  the data has approximately linear structure.

### t-SNE
- **Pros**: Excellent at revealing local cluster structure; widely used for visualization.
- **Cons**: Slow for large datasets; seed-dependent (with `init="random"`); perplexity-sensitive;
  distances between clusters and cluster sizes are not meaningful;
  no transform for new points (a new well needs a refit).
- **Use when**: You want to visually explore high-dimensional data for clusters
  and the dataset is not too large (< 10k-50k points).

### UMAP
- **Pros**: Often fast on large datasets; supports transform of new data (with the same fitted
  scaler).
- **Cons**: Requires an additional package (`umap-learn`); results depend on `n_neighbors`,
  `min_dist`, the metric and the seed; claims that it preserves global structure better than
  t-SNE are debated (the initialization matters); still non-linear and not interpretable.
- **Use when**: You need non-linear dimensionality reduction at scale;
  you need to embed new unseen points.

### Practical Recommendations for Energy Industry Workflows
1. **Always start with PCA** — it is fast and gives you interpretable information
   about feature importance and variance structure.
2. **Use t-SNE or UMAP for visual exploration** — when you need to spot clusters
   or anomalies in high-dimensional well-log or production data; colour by metadata only
   after fitting.
3. **Prefer UMAP over t-SNE** when working with larger datasets or when you need
   to embed new data points into an existing projection.
4. **Combine methods** — use PCA for feature extraction and dimensionality reduction
   as preprocessing, then apply t-SNE/UMAP on the PCA output for faster visualization.
5. **Judge a map by evidence, not by how clean it looks** — trustworthiness at a stated k,
   stability across seeds and settings, and checks in the original logs.

---
## Summary

In this exercise we systematically compared three dimensionality-reduction methods:

1. **PCA** — linear, fast, interpretable, quantifiable retained variance.
2. **t-SNE** — non-linear, excellent local structure, best for small-to-medium visualization tasks.
3. **UMAP** — non-linear, neighbourhood graph, can transform new rows, often fast on large data.

We evaluated each on:
- **Visual quality** — how the facies appear in 2D (labels added after fitting)
- **Computational time** — on this machine; PCA is by far the fastest here
- **Trustworthiness and stability** — neighbourhood preservation at k = 10, across seeds and settings
- **Silhouette score** — how separated the facies look in each map, compared with the 10D value

**Key takeaway**: There is no single best method. PCA is the default starting point;
t-SNE and UMAP are powerful complements for non-linear visualization. The choice
depends on dataset size, analysis goal, and interpretability requirements, and every
map needs its settings, seeds and trustworthiness reported next to it.